# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a comprehensive guide for loading, exploring, and processing the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source

The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their `@id` values.

All navigating (record sets, fields, columns) is performed via their `@id` for reproducibility and clarity.

In [ ]:
# List the available record sets in the dataset
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record sets.")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}, name: {rs.get('name', '')}")

# For each record set, print the available fields with their @id
for rs in record_sets:
    print(f"\nFields in RecordSet {rs['@id']}:" )
    for field in rs.get('field', []):
        print(f"  Field @id: {field['@id']} - {field.get('name', '')}")
        if 'column' in field:
            for column in field['column']:
                print(f"    Column @id: {column['@id']} - {column.get('name', '')}")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. 

Use the record set and field `@id`s from the overview above.

In [ ]:
# Prepare to extract data from each record set using their @id
all_record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in all_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Print available columns in the first record set with data
main_record_set_id = None
for rsid, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rsid
        print(f'RecordSet @id used for analysis: {rsid}')
        print('Columns:', df.columns.tolist())
        display(df.head())
        break
if main_record_set_id is None:
    print("No records found in any record set.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 
This section demonstrates removing outliers, transforming data distributions, and grouping by attributes using only `@id`-based references.

In [ ]:
# Begin EDA on the main record set
df = dataframes[main_record_set_id]

# Let's list all numeric columns and select one by @id
numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        print(f"Numeric field selected (@id): {col}")
        break
if numeric_field_id is None:
    print('No numeric columns found for EDA.')
else:
    threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0
    print(f"\nFiltering rows where {numeric_field_id} > {threshold:.2f}")
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records:")
    display(filtered_df.head())

    # Normalizing the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Attempt to group by a categorical field
    group_field_id = None
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
            group_field_id = col
            print(f'Grouping by categorical field (@id): {col}')
            break

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization

Visualize data distributions or relationships between fields using the main record set.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the main numeric field (if available)
if numeric_field_id is not None and not df[numeric_field_id].isnull().all():
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

# If grouped data exists, plot a bar plot of the means
if 'grouped_df' in locals() and not grouped_df.empty:
    plt.figure(figsize=(10,5))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
    plt.xlabel(group_field_id)
    plt.ylabel(f'Mean {numeric_field_id}')
    plt.title(f'Mean {numeric_field_id} by {group_field_id}')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

We have successfully loaded the FAIR² dataset using the Croissant schema via its URL, explored its record sets and fields by their `@id`, extracted data into DataFrames, and performed initial exploratory data analysis and visualization. 

All fields and entity references were handled using `@id` for reproducibility and clarity. Continue your analysis using `mlcroissant` by referencing the Croissant metadata and schema for richer, more fine-grained semantic exploration.